# Data Cleaning and Preprocessing Pipeline:

**Author:** Mridankan Mandal  
**Roll Number:** IIB2024017

---

## Overview:
- A robust data cleaning pipeline is implemented to prepare raw artwork metadata for downstream modelling.
- The dataset, sourced from a ZIP archive, is extracted and subjected to systematic text normalisation, numeric coercion, dimension recovery, and missing-value imputation.
- All transformations are applied consistently across both the training and test splits to prevent data leakage.
- The cleaned outputs are persisted to `data/processed/` for consumption by the feature-engineering notebook.


## 1. Imports and Directory Setup:
- Standard Python libraries are imported for filesystem operations, archive handling, text processing, and numerical computation.
- The `pathlib.Path` abstraction is used throughout to ensure OS-agnostic path handling.
- Output directories are created idempotently via `os.makedirs(exist_ok=True)` to avoid errors on repeated runs.
- The raw data directory (`data/raw/`) receives extracted CSVs; the processed directory (`data/processed/`) stores cleaned outputs.


In [1]:
import os
from pathlib import Path
import zipfile
import json
import re
import unicodedata
import numpy as np
import pandas as pd

zip_path = Path("classifying-artisticss-mediums-from-metadata.zip")
raw_dir = Path("data/raw")
output_dir = Path("data/processed")

os.makedirs(raw_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

## 2. Archive Extraction:


In [2]:
def extract_archive(zip_path: Path, raw_dir: Path):
    discovered = {}
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        for name in names:
            low = name.lower()
            if low.endswith(".csv") and "train" in low:
                discovered["train"] = name
            elif low.endswith(".csv") and "test" in low and "sample" not in low:
                discovered["test"] = name
            elif low.endswith(".csv") and "sample" in low:
                discovered["sample_submission"] = name

        output_paths = {
            "train": raw_dir / "train.csv",
            "test": raw_dir / "test.csv",
            "sample_submission": raw_dir / "sample_submission.csv",
        }

        for key, archive_name in discovered.items():
            with zf.open(archive_name, "r") as src:
                output_paths[key].write_bytes(src.read())

    return output_paths

output_paths = extract_archive(zip_path, raw_dir)
print("Extracted:", output_paths)

Extracted: {'train': PosixPath('data/raw/train.csv'), 'test': PosixPath('data/raw/test.csv'), 'sample_submission': PosixPath('data/raw/sample_submission.csv')}


## 3. Text Normalisation and Unicode Remediation:
- All text columns are passed through a multi-stage Unicode cleaning pipeline.
- NFKC normalisation is first applied to decompose compatibility characters (ligatures, em-dashes) into their canonical equivalents.
- A curated translation table maps visually ambiguous glyphs: smart quotes, dashes, ellipses, multiplication signs, and common copyright symbols: to their plain ASCII counterparts.
- Invisible Unicode control characters (zero-width spaces, directional markers, word-joiner characters) are stripped via a compiled regular expression.
- Non-breaking spaces (\xa0) are replaced with ordinary spaces before a final NFKD pass that removes combining diacritics.
- The resulting string is ASCII encoded, and excess whitespace is collapsed, yielding a clean, model-ready token sequence.
- This normalisation is applied to all explicitly listed text columns as well as any remaining object-dtype columns discovered at runtime.


In [3]:
TEXT_COLUMNS = [
    "t", "txt", "cap", "dim", "cat", "classification", "inscription",
    "markings", "attribution", "attributioninverted", "provenancetext",
    "visualbrowserclassification", "departmentabbr", "note", "tag", "ts", "dt"
]

UNICODE_TRANSLATIONS = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201C": '"', "\u201D": '"',
    "\u2013": "-", "\u2014": "-", "\u2026": "...", "\u00D7": "x",
    "\u2212": "-", "\u00BD": "1/2", "\u00BC": "1/4", "\u00BE": "3/4",
    "\u00A9": "(c)", "\u00AE": "(r)", "\u2122": "tm", "\u03A3": "S",
})
INVISIBLE_CHAR_RE = re.compile(r"[\u200B-\u200F\u202A-\u202E\u2060-\u206F\uFEFF]")

def normalize_whitespace(value: object) -> str:
    if pd.isna(value):
        return ""
    text = str(value)
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(UNICODE_TRANSLATIONS)
    text = INVISIBLE_CHAR_RE.sub("", text)
    text = text.replace("\xa0", " ")
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def coerce_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    preferred_cols = [col for col in TEXT_COLUMNS if col in out.columns]
    normalized_cols = set(preferred_cols)

    for col in preferred_cols:
        out[col] = out[col].map(lambda v: normalize_whitespace(v))

    for col in out.columns:
        if col in normalized_cols:
            continue
        if out[col].dtype == object:
            out[col] = out[col].map(lambda v: normalize_whitespace(v))
    return out


## 4. Numeric Coercion and Dimension Recovery:
- Columns designated as numeric (y0, y1, width, height, dimension) are cast to float using `pd.to_numeric(..., errors='coerce')`; non-parseable values silently become NaN rather than raising exceptions.
- The `dim` free-text column is parsed to recover physical dimensions when `width` or `height` are absent.
- A two-pattern regex strategy is employed: the first matches the standard `HxW` format; the second handles more explicit `h. H x w. W` notation.
- Parsed dimension pairs are back-filled only into truly-missing cells, ensuring that originally-present numeric values are not overwritten.
- Strictly positive pairs are required (both dimensions > 0) before being accepted, guarding against degenerate dimension strings.


In [4]:
NUMERIC_COLUMNS = ["y0", "y1", "width", "height", "dimension"]

def coerce_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in NUMERIC_COLUMNS:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

def parse_dim_pair(dim_text: str) -> tuple:
    if not dim_text:
        return (np.nan, np.nan)
    lowered = dim_text.lower().replace(",", ".").replace("×", "x")
    patterns = [
        r"([0-9]+(?:\.[0-9]+)?)\s*x\s*([0-9]+(?:\.[0-9]+)?)",
        r"h\.?\s*([0-9]+(?:\.[0-9]+)?)\s*[x]\s*w\.?\s*([0-9]+(?:\.[0-9]+)?)",
    ]
    for pattern in patterns:
        match = re.search(pattern, lowered)
        if match:
            a, b = float(match.group(1)), float(match.group(2))
            if a > 0 and b > 0:
                return (a, b)
    return (np.nan, np.nan)

def recover_dimensions(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "dim" not in out.columns:
        return out
    width_missing = out["width"].isna() if "width" in out.columns else pd.Series(False, index=out.index)
    height_missing = out["height"].isna() if "height" in out.columns else pd.Series(False, index=out.index)
    needs_parse = width_missing | height_missing
    parsed = out.loc[needs_parse, "dim"].map(parse_dim_pair)
    parsed_df = pd.DataFrame(parsed.tolist(), index=parsed.index, columns=["parsed_w", "parsed_h"])
    if "width" in out.columns:
        mask = out["width"].isna() & parsed_df["parsed_w"].notna()
        out.loc[mask.index[mask], "width"] = parsed_df.loc[mask.index[mask], "parsed_w"]
    if "height" in out.columns:
        mask = out["height"].isna() & parsed_df["parsed_h"].notna()
        out.loc[mask.index[mask], "height"] = parsed_df.loc[mask.index[mask], "parsed_h"]
    return out

## 5. Missing Value Imputation:
- Median imputation is chosen for numeric columns because it is robust to the skewed distributions typical of artwork metadata (like century spanning date ranges, highly variable physical sizes).
- Medians are fitted exclusively on the training split via `fit_impute_values`, preventing any flow of test-set statistics back into the training pipeline, a common source of data leakage.
- For each imputed column a binary missingness indicator (`{col}_was_missing`) is appended as an `int8` flag, preserving the information that a value was originally absent for downstream models to exploit.
- The fitted imputation map is then applied identically to both splits through `apply_imputation`, ensuring consistent handling of unseen missing values at inference time.


In [5]:
def fit_impute_values(train_df: pd.DataFrame) -> dict:
    values = {}
    for col in NUMERIC_COLUMNS:
        if col in train_df.columns:
            series = train_df[col]
            median = float(series.median()) if series.notna().any() else 0.0
            values[col] = median
    return values

def apply_imputation(df: pd.DataFrame, impute_values: dict) -> pd.DataFrame:
    out = df.copy()
    for col, value in impute_values.items():
        if col in out.columns:
            out[f"{col}_was_missing"] = out[col].isna().astype(np.int8)
            out[col] = out[col].fillna(value)
    return out

## 6. Cleaning Pipeline Execution:
- The master `clean_dataframe` function orchestrates the full sequence: column-name stripping, unnamed-index column removal, duplicate ID deduplication (keeping the first occurrence), text normalisation, numeric coercion, and dimension recovery.
- Both the training and test DataFrames are independently cleaned through this function to ensure identical column schemas.
- The imputation map is then fitted on the cleaned training frame and applied to both splits.
- Final shapes are reported to confirm that no rows have been inadvertently dropped and that the expected feature count is present.
- All three outputs (cleaned train, cleaned test, sample submission) are persisted to `data/processed/` as CSV files for consumption by the feature engineering notebook.


In [6]:
def drop_useless_index_columns(df: pd.DataFrame) -> pd.DataFrame:
    drop_cols = [c for c in df.columns if c.lower().startswith("unnamed:")]
    return df.drop(columns=drop_cols) if drop_cols else df.copy()

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    out = drop_useless_index_columns(out)
    if "id" in out.columns:
        out = out.drop_duplicates(subset=["id"], keep="first").copy()
    out = coerce_text_columns(out)
    out = coerce_numeric_columns(out)
    out = recover_dimensions(out)
    return out

# Load datasets
train_raw = pd.read_csv(output_paths["train"], low_memory=False)
test_raw = pd.read_csv(output_paths["test"], low_memory=False)
sample_sub = pd.read_csv(output_paths["sample_submission"], low_memory=False)

# Clean DataFrames
print("Cleaning train...")
train_clean = clean_dataframe(train_raw)
print("Cleaning test...")
test_clean = clean_dataframe(test_raw)

# Null Imputation Pipeline
print("Handling Imputations...")
impute_map = fit_impute_values(train_clean)
train_clean = apply_imputation(train_clean, impute_map)
test_clean = apply_imputation(test_clean, impute_map)

print("Train Shape After Cleaning:", train_clean.shape)
print("Test Shape After Cleaning:", test_clean.shape)

# Save outputs
print("Saving processes chunks...")
train_clean.to_csv(output_dir / "train_clean.csv", index=False)
test_clean.to_csv(output_dir / "test_clean.csv", index=False)
sample_sub.to_csv(output_dir / "sample_submission.csv", index=False)
print("Data cleaning is complete! Files saved in `data/processed/`.")


Cleaning train...


Cleaning test...
Handling Imputations...
Train Shape After Cleaning: (4000, 61)
Test Shape After Cleaning: (1000, 60)
Saving processes chunks...


Data cleaning is complete! Files saved in `data/processed/`.


---

**Thank You for reading this.**
